In [32]:
# Baseline model class

import pandas as pd
import os
import joblib
import pickle
import json
from sklearn import model_selection, preprocessing, metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar
from typing import Dict

class TFIDFBaselineModel:
    def __init__(self, training_set, validation_set, text_column: str, label_column: str):
        self.training_set = training_set
        self.validation_set = validation_set
        self.text_column = text_column
        self.label_column = label_column
        self.vectorizer = TfidfVectorizer()
        self.label_encoder = LabelEncoder()
        self.model = LogisticRegression(max_iter=1000)

    def preprocess_data(self):
        # Fit the TF-IDF vectorizer on the training data and transform both training and validation data

        self.X_train = self.vectorizer.fit_transform(self.training_set[self.text_column])
        self.X_validation = self.vectorizer.transform(self.validation_set[self.text_column])

        # Encode the labels
        self.y_train = self.label_encoder.fit_transform(self.training_set[self.label_column])
        self.y_validation = self.label_encoder.transform(self.validation_set[self.label_column])

    def train_model(self):
        # Train the logistic regression model
        self.model.fit(self.X_train, self.y_train)

    def evaluate_model(self):
        # Make predictions on the validation set
        y_pred = self.model.predict(self.X_validation)

        # Calculate accuracy
        accuracy = metrics.accuracy_score(self.y_validation, y_pred)
        classification_report = metrics.classification_report(self.y_validation, y_pred, target_names=self.label_encoder.classes_)
        brier = metrics.brier_score_loss(self.y_validation, self.model.predict_proba(self.X_validation)[:, 1])
        print(f'Validation Accuracy: {accuracy:.4f}')
        print(f'Classification Report: \n{classification_report}')
        print(f'Validation Brier score: {brier:.4f}')

    def run_pipeline(self):
        self.preprocess_data()
        self.train_model()
        self.evaluate_model()

    def save_model(self, output_dir):
        # Save the trained model, vectorizer, and label encoder to the specified output directory
        os.makedirs(output_dir, exist_ok=True)
        artifacts = {
                    'model': self.model,
                    'vectorizer': self.vectorizer,
                    'encoder': self.label_encoder
                    }
                
        filepath = os.path.join(output_dir, 'uncalibrated_meta.pkl')
        with open(filepath, 'wb') as f:
            pickle.dump(artifacts, f)
        print(f'Uncalibrated model artifacts saved to {filepath}')

    def save_metrics(self, output_dir, output_format='txt'):
        # Save the evaluation metrics to a text file in the specified output directory
        os.makedirs(output_dir, exist_ok=True)
        y_pred = self.model.predict(self.X_validation)
        accuracy = metrics.accuracy_score(self.y_validation, y_pred)
        classification_report = metrics.classification_report(self.y_validation, y_pred, target_names=self.label_encoder.classes_)
        if output_format == 'txt':
            with open(os.path.join(output_dir, 'metrics.txt'), 'w') as f:
                f.write(f'Validation Accuracy: {accuracy:.4f}\n')
                f.write(f'Classification Report: \n{classification_report}\n')
        elif output_format == 'json':
            with open(os.path.join(output_dir, 'metrics.json'), 'w') as f:
                json.dump({
                    'validation_accuracy': accuracy,
                    'classification_report': classification_report
                }, f)


        

In [30]:
training = pd.read_csv('../../data/splits/training_50_50.csv')
validation = pd.read_csv('../../data/splits/validation_50_50.csv')

baseline_model = TFIDFBaselineModel(training, validation, text_column='Full_Text', label_column='Label')

baseline_model.run_pipeline()


Validation Accuracy: 0.9573
Classification Report: 
              precision    recall  f1-score   support

      Benign       0.95      0.96      0.96      2000
   Malicious       0.96      0.95      0.96      2000

    accuracy                           0.96      4000
   macro avg       0.96      0.96      0.96      4000
weighted avg       0.96      0.96      0.96      4000

Validation Brier score: 0.0421


In [10]:
baseline_model.save_metrics(output_dir='../../experiments/binary/baseline/uncalibrated', output_format='txt')

In [33]:
baseline_model.save_model(output_dir='../../models/binary/baseline/uncalibrated')

Uncalibrated model artifacts saved to ../../models/binary/baseline/uncalibrated\uncalibrated_meta.pkl
Model, vectorizer, and label encoder saved to ../../models/binary/baseline/uncalibrated


In [34]:
import os
import joblib
import pickle
import numpy as np
from scipy.optimize import minimize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV

# A class for calibrating the baseline model using temperature scaling

class BaselineCalibrator:
    def __init__(self, model, vectorizer, encoder, calibration_set, validation_set, text_column: str, label_column: str, temperature: float = 1.0):
        self.original_model = model
        self.calibration_set = calibration_set
        self.validation_set = validation_set
        self.text_column = text_column
        self.label_column = label_column
        self.vectorizer = vectorizer
        self.label_encoder = encoder
        self.temperature = temperature

    def preprocess_data(self):
        # Fit the TF-IDF vectorizer on the calibration data and transform both calibration and validation data
        self.X_calibration = self.vectorizer.transform(self.calibration_set[self.text_column])
        self.X_validation = self.vectorizer.transform(self.validation_set[self.text_column])

        # Encode the labels
        self.y_calibration = self.label_encoder.transform(self.calibration_set[self.label_column])
        self.y_validation = self.label_encoder.transform(self.validation_set[self.label_column])

    def _get_logits(self, X):
        # Extract raw logits (before sigmoid/softmax conversion) from LogisticRegression
        # decision_function returns: w * X + b
        return self.original_model.decision_function(X)

    def calibrate(self):
        # Optimize the temperature parameter to minimize the loss
        raw_logits = self._get_logits(self.X_calibration)

        def objective(T):
            # Prevent division by zero
            T = max(T[0], 1e-5) 
            
            # Apply temperature scaling to logits
            scaled_logits = raw_logits / T
            
            # Convert scaled logits to probabilities using sigmoid
            probs = 1 / (1 + np.exp(-scaled_logits))
            
            # Return log loss against true labels
            return log_loss(self.y_calibration, probs)

        # 3. Optimize the temperature parameter starting at T=1.0
        result = minimize(objective, x0=[1.0], method='Nelder-Mead')
        self.temperature = max(result.x[0], 1e-5)
        print(f"Optimized Temperature: {self.temperature:.4f}")

    def predict_proba(self, X):
        # Predict probabilities using the calibrated model
        raw_logits = self._get_logits(X)
        scaled_logits = raw_logits / self.temperature
        probs = 1 / (1 + np.exp(-scaled_logits))
        return np.vstack([1 - probs, probs]).T  # Return as a 2D array with shape (n_samples, 2)

    def evaluate_calibration(self):
        # Evaluate the calibrated model on the validation set
        scaled_logits = self.predict_proba(self.X_validation)
        loss = log_loss(self.y_validation, scaled_logits)
        brier = brier_score_loss(self.y_validation, scaled_logits[:, 1])
        accuracy = np.mean(np.argmax(scaled_logits, axis=1) == self.y_validation)
        classification_report = metrics.classification_report(self.y_validation, np.argmax(scaled_logits, axis=1), target_names=self.label_encoder.classes_)

        print(f'Validation Loss after calibration: {loss:.4f}')
        print(f'Validation Brier score after calibration: {brier:.4f}')
        print(f'Validation Accuracy after calibration: {accuracy:.4f}')
        print(f'Classification Report after calibration: \n{classification_report}')

    def run_calibration_pipeline(self):
        self.preprocess_data()
        self.calibrate()
        self.evaluate_calibration()

    def save_calibrated_model(self, output_dir):
        # Save both the original model structure and the optimal temperature setting
        artifacts = {
            'model': self.original_model,
            'temperature': self.temperature,
            'vectorizer': self.vectorizer,
            'encoder': self.label_encoder
        }
        
        filepath = os.path.join(output_dir, 'temperature_calibrated_meta.pkl')
        with open(filepath, 'wb') as f:
            pickle.dump(artifacts, f)
        print(f'Calibrated model artifacts saved to {filepath}')
        


    

In [37]:
model_artifacts_path = '../../models/binary/baseline/uncalibrated/uncalibrated_meta.pkl'

model_data = pickle.load(open(model_artifacts_path, 'rb'))

model = model_data['model']
vectorizer = model_data['vectorizer']
encoder = model_data['encoder']

calibration_set = pd.read_csv('../../data/splits/calibration_80_20.csv')
validation_set = pd.read_csv('../../data/splits/validation_80_20.csv')

calibrator = BaselineCalibrator(model=model, vectorizer=vectorizer, encoder=encoder, calibration_set=calibration_set, validation_set=validation_set, text_column='Full_Text', label_column='Label')

calibrator.run_calibration_pipeline()

Optimized Temperature: 0.4121
Validation Loss after calibration: 0.1059
Validation Brier score after calibration: 0.0277
Validation Accuracy after calibration: 0.9675
Classification Report after calibration: 
              precision    recall  f1-score   support

      Benign       0.99      0.97      0.98      3200
   Malicious       0.89      0.96      0.92       800

    accuracy                           0.97      4000
   macro avg       0.94      0.97      0.95      4000
weighted avg       0.97      0.97      0.97      4000



In [28]:
calibrator.save_calibrated_model(output_dir='../../models/binary/baseline/calibrated')

Calibrated model artifacts saved to ../../models/binary/baseline/calibrated\temperature_calibrated_meta.pkl


In [1]:
# BERT Model Class and TextDataset Class

import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

class TextDataset(Dataset):

    def __init__(self, dataset, mode, max_len):
        self.texts = dataset['Full_Text'].values
        self.labels = dataset['Label'].values
        self.encoder = LabelEncoder()
        self.tokenizer = AutoTokenizer.from_pretrained(mode)
        self.max_len = max_len
        self.attention_mask = None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        if self.tokenizer:

            encoding = self.tokenizer.encode_plus(
                text,
                add_special_tokens=True,
                max_length=self.max_len,
                return_token_type_ids=False,
                padding='max_length',
                truncation=True,
                return_attention_mask=True,
                return_tensors='pt',
            )

            return {
                'text': text,
                'input_ids': encoding['input_ids'].flatten(),
                'attention_mask': encoding['attention_mask'].flatten(),
                'labels': torch.tensor(label, dtype=torch.long)
            }

        return {'text': text, 'labels': torch.tensor(label, dtype=torch.long)}

class BERTClassifier(nn.Module):
    def __init__(self, n_classes, train_loader, val_loader, pretrained_model_name='bert-base-cased'):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(pretrained_model_name)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.temp_scalar = None

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs[1]
        output = self.drop(pooled_output)
        return self.out(output)

    def train(self, device, optimizer, criterion, epochs):
        
        for epoch in range(epochs):
            total_loss = 0
            for batch in self.train_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                optimizer.zero_grad()
                outputs = self(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            validation_loss = 0
            with torch.no_grad():
                for batch in self.val_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)

                    outputs = self(input_ids, attention_mask)
                    loss = criterion(outputs, labels)
                    validation_loss += loss.item()

            print(f'Epoch {epoch + 1}/{epochs}, Training Loss: {total_loss / len(self.train_loader)}, Validation Loss: {validation_loss / len(self.val_loader)}')
            
        return total_loss / len(self.train_loader), validation_loss / len(self.val_loader)


In [4]:
import pandas as pd

data = pd.read_csv('../../data/splits/training_50_50.csv')

dataset = TextDataset(data, mode='google-bert/bert-base-cased', max_len=1024)

data_loader = DataLoader(dataset, batch_size=36, shuffle=True)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

c:\Users\numbe\OneDrive\Desktop\AIML339 Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\numbe\.cache\huggingface\hub\models--google-bert--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Temperature Scaling Class for Calibration Set

class TemperatureScaling(nn.Module):
    def __init__(self):
        super(TemperatureScaling, self).__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    